# **Boids Part 1 - Simulation, Visualization and Pygame**

## **Table of Contents**

1. [Introduction to Boids](#introduction)
2. [Setting Up the Environment](#setup)
3. [Basic Boids Implementation in Python](#basic_implementation)
    - [3.1 Defining the Boid Class](#boid_class)
    - [3.2 Implementing Flocking Rules](#flocking_rules)
4. [Visualizing Boids with Matplotlib](#matplotlib_visualization)
    - [4.1 Setting Up the Plot](#matplotlib_setup)
    - [4.2 Running the Simulation](#matplotlib_simulation)
5. [Introducing Pygame for Interactive Simulation](#pygame_intro)
    - [5.1 Setting Up a Basic Pygame Window](#pygame_setup)
    - [5.2 Adding Boids to Pygame](#pygame_boids)
6. [Enhancing the Pygame Simulation](#enhancements)
    - [6.1 Implementing Flocking Behavior](#flocking_behavior)
    - [6.2 Adding Boundary Conditions](#boundary_conditions)
    - [6.3 Interactive Elements](#interactive_elements)
7. [Exercises and Exploration](#exercises)
8. [Conclusion](#conclusion)
9. [References](#references)

Misc: [Setting up your Local Python Environment](#environment_setup)


## <a name="introduction"></a>1. Introduction to Boids

**Boids** is an artificial life program developed by Craig Reynolds in 1986 to simulate the flocking behavior of birds. The Boids model demonstrates how complex group behaviors can emerge from simple individual rules. Boids are agents that follow three primary rules:

1. **Separation**: Avoid crowding neighbors (steer to avoid too close proximity).
2. **Alignment**: Align velocity with nearby Boids (steer towards the average heading of neighbors).
3. **Cohesion**: Move towards the average position of neighbors (steer to move towards the average position).

These simple rules result in realistic flocking behavior without centralized control.

**Applications**:
- Animation and Visual Effects
- Robotics (flocking drones)
- Video Games
- Simulation of natural phenomena

In this lab, we will implement a basic Boids simulation in Python, visualize it using Matplotlib, and create an interactive simulation using Pygame.


## <a name="setup"></a>2. Setting Up the Environment

Before we begin, ensure that you have the necessary Python libraries installed. We will use `numpy` for numerical operations, `matplotlib` for basic visualization, and `pygame` for interactive simulations.

### **Installation Instructions**

If you're using a local Python environment, you can install the required libraries using `pip`. Execute the following commands in your terminal or command prompt:

```bash
pip install numpy matplotlib pygame


## <a name="basic_implementation"></a>3. Basic Boids Implementation in Python

### <a name="boid_class"></a>3.1 Defining the Boid Class

We start by defining a `Boid` class that represents each individual agent in the simulation. Each Boid has a position and a velocity, both represented as 2D vectors.

We will be using Python libraries, including matplotlib for basic visualization and pygame (on your pc) for interactive graphics.


In [ ]:
!pip install pygame matplotlib numpy

In [2]:
import numpy as np

class Boid:
    def __init__(self, position, velocity):
        """
        Initialize a Boid with a position and velocity.

        :param position: Initial position as a numpy array [x, y]
        :param velocity: Initial velocity as a numpy array [vx, vy]
        """
        self.position = np.array(position, dtype='float64')
        self.velocity = np.array(velocity, dtype='float64')

    def update_position(self, width, height):
        """
        Update the Boid's position based on its velocity.
        Wrap around the edges to keep Boids within bounds.

        :param width: Width of the simulation area
        :param height: Height of the simulation area
        """
        self.position += self.velocity

        # Wrap around the edges
        self.position[0] = self.position[0] % width
        self.position[1] = self.position[1] % height


### <a name="flocking_rules"></a>3.2 Implementing Flocking Rules

Next, we implement the three primary flocking rules: Separation, Alignment, and Cohesion. Each rule calculates a steering vector that influences the Boid's velocity.


In [3]:
class Boid:
    def __init__(self, position, velocity):
        """
        Initialize a Boid with a position and velocity.

        :param position: Initial position as a numpy array [x, y]
        :param velocity: Initial velocity as a numpy array [vx, vy]
        """
        self.position = np.array(position, dtype='float64')
        self.velocity = np.array(velocity, dtype='float64')

    def update_position(self, width, height):
        """
        Update the Boid's position based on its velocity.
        Wrap around the edges to keep Boids within bounds.

        :param width: Width of the simulation area
        :param height: Height of the simulation area
        """
        self.position += self.velocity

        # Wrap around the edges
        self.position[0] = self.position[0] % width
        self.position[1] = self.position[1] % height

    def separation(self, boids, separation_distance=20):
        """
        Calculate the separation steering vector.

        :param boids: List of all Boids in the simulation
        :param separation_distance: Minimum desired distance between Boids
        :return: Separation vector as a numpy array
        """
        steer = np.zeros(2)
        count = 0
        for other in boids:
            distance = np.linalg.norm(self.position - other.position)
            if 0 < distance < separation_distance:
                steer += self.position - other.position
                count += 1
        if count > 0:
            steer /= count
        return steer

    def alignment(self, boids, neighbor_distance=50):
        """
        Calculate the alignment steering vector.

        :param boids: List of all Boids in the simulation
        :param neighbor_distance: Distance to consider for alignment
        :return: Alignment vector as a numpy array
        """
        avg_velocity = np.zeros(2)
        count = 0
        for other in boids:
            distance = np.linalg.norm(self.position - other.position)
            if 0 < distance < neighbor_distance:
                avg_velocity += other.velocity
                count += 1
        if count > 0:
            avg_velocity /= count
            return avg_velocity - self.velocity
        return np.zeros(2)

    def cohesion(self, boids, neighbor_distance=50):
        """
        Calculate the cohesion steering vector.

        :param boids: List of all Boids in the simulation
        :param neighbor_distance: Distance to consider for cohesion
        :return: Cohesion vector as a numpy array
        """
        center_of_mass = np.zeros(2)
        count = 0
        for other in boids:
            distance = np.linalg.norm(self.position - other.position)
            if 0 < distance < neighbor_distance:
                center_of_mass += other.position
                count += 1
        if count > 0:
            center_of_mass /= count
            return center_of_mass - self.position
        return np.zeros(2)

    def apply_behaviors(self, boids,
                        separation_weight=1.5,
                        alignment_weight=1.0,
                        cohesion_weight=1.0,
                        max_speed=10):
        """
        Apply the three flocking behaviors to update velocity.

        :param boids: List of all Boids in the simulation
        :param separation_weight: Weight for separation behavior
        :param alignment_weight: Weight for alignment behavior
        :param cohesion_weight: Weight for cohesion behavior
        :param max_speed: Maximum speed limit for velocity
        """
        sep = self.separation(boids) * separation_weight
        ali = self.alignment(boids) * alignment_weight
        coh = self.cohesion(boids) * cohesion_weight
        self.velocity += sep + ali + coh
        self.limit_speed(max_speed)

    def limit_speed(self, max_speed):
        """
        Limit the Boid's speed to the specified maximum.

        :param max_speed: Maximum allowed speed
        """
        speed = np.linalg.norm(self.velocity)
        if speed > max_speed:
            self.velocity = (self.velocity / speed) * max_speed


---

## <a name="matplotlib_visualization"></a>4. Visualizing Boids with Matplotlib

Visualization helps in understanding the behavior of Boids. We'll use Matplotlib to create an animated scatter plot showing the positions of Boids over time.


### <a name="matplotlib_setup"></a>4.1 Setting Up the Plot

We'll use `FuncAnimation` from `matplotlib.animation` to create a smooth animation of Boids.


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

# Define the simulation area
WIDTH, HEIGHT = 100, 100

# Initialize Boids
num_boids = 20
boids = [
    Boid(position=np.random.rand(2) * [WIDTH, HEIGHT],
         velocity=np.random.rand(2) * 10 - 5)
    for _ in range(num_boids)
]

# Set up the figure and axis
fig, ax = plt.subplots(figsize=(8, 8))
ax.set_xlim(0, WIDTH)
ax.set_ylim(0, HEIGHT)
ax.set_title('Boids Simulation with Matplotlib')
scatter = ax.scatter([boid.position[0] for boid in boids],
                     [boid.position[1] for boid in boids],
                     color='blue')


### <a name="matplotlib_simulation"></a>4.2 Running the Simulation

Define an update function that will be called for each frame of the animation.


In [5]:
def animate(frame_num):
    """
    Update function for the animation.

    :param frame_num: Current frame number (unused)
    :return: Updated scatter plot
    """
    for boid in boids:
        boid.apply_behaviors(boids)
        boid.update_position(WIDTH, HEIGHT)

    # Update scatter plot data
    scatter.set_offsets([boid.position for boid in boids])
    return scatter,

# Create the animation
ani = FuncAnimation(fig, animate, frames=200, interval=50, blit=True)

# Display the animation
plt.show()


**Notes**:
- **FuncAnimation**: Efficiently updates the plot by only redrawing the elements that have changed.
- **blit=True**: Optimizes the drawing by only updating the parts of the plot that have changed.
- **Interval**: Controls the speed of the animation (in milliseconds).

**Running in Google Colab**:
If you're using Google Colab, standard `plt.show()` may not render the animation properly. Instead, use the following to display the animation as HTML:

In [ ]:
from IPython.display import HTML
HTML(ani.to_jshtml())


## <a name="pygame_intro"></a>5. Introducing Pygame for Interactive Simulation

While Matplotlib provides basic visualization, Pygame allows for more interactive and real-time simulations. Pygame is a set of Python modules designed for writing video games, but it's also suitable for simulations requiring real-time graphics and user interaction.

**Important**: Pygame is best run on a local machine. It may not work as expected in online environments like Google Colab.

See [Setting Up Your Python Environment](#environment_setup) if in need of assistance to get python running on your pc.

### <a name="pygame_setup"></a>5.1 Setting Up a Basic Pygame Window

First, we'll set up a basic Pygame window to display our simulation.


In [ ]:
import pygame
import numpy as np

# Initialize Pygame
pygame.init()

# Define simulation window dimensions
WINDOW_WIDTH, WINDOW_HEIGHT = 800, 600
screen = pygame.display.set_mode((WINDOW_WIDTH, WINDOW_HEIGHT))
pygame.display.set_caption('Boids Simulation with Pygame')

# Define clock to control frame rate
clock = pygame.time.Clock()
FPS = 60  # Frames per second

# Define colors
BLACK = (0, 0, 0)
WHITE = (255, 255, 255)

# Minimal Pygame loop to keep the window open
running = True
while running:
    clock.tick(FPS)  # Maintain frame rate

    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

    screen.fill(BLACK)  # Fill the screen with black
    pygame.display.flip()  # Update the display

pygame.quit()


### <a name="pygame_boids"></a>5.2 Adding Boids to Pygame

Now, we'll modify our Boid class to work within the Pygame environment and visualize the Boids as circles.


In [ ]:
import numpy as np

class Boid:
    def __init__(self, position, velocity):
        self.position = np.array(position, dtype='float64')
        self.velocity = np.array(velocity, dtype='float64')

    def update_position(self, width, height):
        self.position += self.velocity
        self.position[0] = self.position[0] % width
        self.position[1] = self.position[1] % height

    def separation(self, boids, separation_distance=20):
        steer = np.zeros(2)
        count = 0
        for other in boids:
            distance = np.linalg.norm(self.position - other.position)
            if 0 < distance < separation_distance:
                steer += self.position - other.position
                count += 1
        if count > 0:
            steer /= count
        return steer

    def alignment(self, boids, neighbor_distance=50):
        avg_velocity = np.zeros(2)
        count = 0
        for other in boids:
            distance = np.linalg.norm(self.position - other.position)
            if 0 < distance < neighbor_distance:
                avg_velocity += other.velocity
                count += 1
        if count > 0:
            avg_velocity /= count
            return avg_velocity - self.velocity
        return np.zeros(2)

    def cohesion(self, boids, neighbor_distance=50):
        center_of_mass = np.zeros(2)
        count = 0
        for other in boids:
            distance = np.linalg.norm(self.position - other.position)
            if 0 < distance < neighbor_distance:
                center_of_mass += other.position
                count += 1
        if count > 0:
            center_of_mass /= count
            return center_of_mass - self.position
        return np.zeros(2)

    def apply_behaviors(self, boids, separation_weight=1.5, alignment_weight=1.0, cohesion_weight=1.0, max_speed=2):
        sep = self.separation(boids) * separation_weight
        ali = self.alignment(boids) * alignment_weight
        coh = self.cohesion(boids) * cohesion_weight
        self.velocity += sep + ali + coh
        self.limit_speed(max_speed)

    def limit_speed(self, max_speed):
        speed = np.linalg.norm(self.velocity)
        if speed > max_speed:
            self.velocity = (self.velocity / speed) * max_speed

    def draw(self, surface):
        pygame.draw.circle(surface, WHITE, self.position.astype(int), 3)


In [ ]:
# Initialize Boids for Pygame
num_boids = 30
boids_pygame = [
    Boid(position=np.random.rand(2) * [WINDOW_WIDTH, WINDOW_HEIGHT],
         velocity=np.random.rand(2))
    for _ in range(num_boids)
]


---

## <a name="enhancements"></a>6. Enhancing the Pygame Simulation

To create a more realistic and interactive simulation, we'll integrate the flocking behaviors and add boundary conditions.


### <a name="flocking_behavior"></a>6.1 Implementing Flocking Behavior

Ensure that each Boid applies the flocking behaviors before updating its position.


In [ ]:
# Main Pygame loop
running = True
while running:
    clock.tick(FPS)  # Maintain frame rate

    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

    # Fill the background
    screen.fill(BLACK)

    # Update and draw each Boid
    for boid in boids_pygame:
        boid.apply_behaviors(boids_pygame)
        boid.update_position(WINDOW_WIDTH, WINDOW_HEIGHT)
        boid.draw(screen)

    # Update the display
    pygame.display.flip()

pygame.quit()


### <a name="boundary_conditions"></a>6.2 Adding Boundary Conditions

Instead of wrapping around the edges, Boids can bounce off the window boundaries. We'll modify the `update_position` method to include boundary reflection.


In [ ]:
def update_position(self, width, height, radius=3):
    # move
    self.position += self.velocity

    # bounce on X (respect radius and clamp inside)
    if self.position[0] < radius:
        self.position[0] = radius
        self.velocity[0] *= -1
    elif self.position[0] > width - radius:
        self.position[0] = width - radius
        self.velocity[0] *= -1

    # bounce on Y (respect radius and clamp inside)
    if self.position[1] < radius:
        self.position[1] = radius
        self.velocity[1] *= -1
    elif self.position[1] > height - radius:
        self.position[1] = height - radius
        self.velocity[1] *= -1

### <a name="interactive_elements"></a>6.3 Interactive Elements

Enhance the simulation by adding interactive elements, such as making Boids respond to mouse movements.

**Example**: Attract Boids towards the mouse position when the left mouse button is pressed.


In [ ]:
# Modify the main loop to include mouse interaction
running = True
while running:
    clock.tick(FPS)

    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

    # Get mouse position and state
    mouse_pos = np.array(pygame.mouse.get_pos(), dtype='float64')
    mouse_pressed = pygame.mouse.get_pressed()[0]  # Left button

    # Fill the background
    screen.fill(BLACK)

    for boid in boids_pygame:
        if mouse_pressed:
            direction = mouse_pos - boid.position
            if np.linalg.norm(direction) != 0:
                boid.velocity += direction * 0.001  # Adjust strength as needed

        boid.apply_behaviors(boids_pygame)
        boid.update_position(WINDOW_WIDTH, WINDOW_HEIGHT)
        boid.draw(screen)

    pygame.display.flip()

pygame.quit()

**Explanation**:
- **Mouse Interaction**: When the left mouse button is pressed, Boids are attracted towards the mouse position by slightly adjusting their velocities.
- **Attraction Strength**: The factor `0.001` controls how strongly Boids are attracted. Adjust this value to change the responsiveness.

*Press and hold the left mouse button.
Boids should start moving towards the mouse cursor, altering their flocking behavior dynamically.
Release the mouse button to return to normal flocking. italicized text*

### <a name="speed_color"></a>6.4 Color by speed

Implement a feature where Boids change color based on their speed. Faster Boids will appear red, while slower ones will appear blue.

* Modify the draw method in the Boid class to change the Boid's color based on its current speed.


In [2]:
# Modify the draw method to color Boids based on their speed
def draw(self, surface, max_speed):
    """
    Draw the Boid as a circle, colored based on its speed.
    Fast Boids (near max_speed) appear red, slower ones appear blue.
    
    :param surface: Pygame surface to draw on
    :param max_speed: Maximum speed for color comparison
    """
    speed = np.linalg.norm(self.velocity)
    
    # Define colors
    RED = (255, 0, 0)
    BLUE = (0, 0, 255)
    
    # Color based on speed: red if near max speed, blue otherwise
    if speed > max_speed * 0.99:
        pygame.draw.circle(surface, RED, self.position.astype(int), 3)
    else:
        pygame.draw.circle(surface, BLUE, self.position.astype(int), 3)

---

## <a name="exercises"></a>7. Exercises and Exploration

To deepen your understanding and extend the simulation, try the following exercises:

1. **Experiment with Rule Weights**:
    - Adjust the `separation_weight`, `alignment_weight`, and `cohesion_weight` in the `apply_behaviors` method.
    - Observe how changing these weights affects the flocking behavior.

2. **Add Obstacles**:
    - Introduce static obstacles in the simulation area.
    - Modify the Boid's behavior to avoid these obstacles using a repulsion vector.

3. **Variable Speeds**:
    - Allow Boids to have variable speeds.
    - Implement acceleration and deceleration based on local flock density.

4. **Obstacle Avoidance**:
    - Add dynamic obstacles that move within the simulation.
    - Implement collision detection and avoidance behaviors.

5. **Leader Boids**:
    - Designate certain Boids as leaders that others follow.
    - Observe how leadership influences the flock.

6. **Interactive Controls**:
    - Add keyboard controls to adjust simulation parameters in real-time (e.g., increase/decrease separation distance).

7. **3D Simulation**:
    - Extend the simulation to three dimensions using libraries like `pygame` with 3D support or `matplotlib`'s 3D plotting capabilities.


## <a name="conclusion"></a>8. Conclusion

In this lab, we've explored the fundamentals of Boids simulation, implementing and visualizing flocking behavior using Python. Starting with a basic Boid class, we integrated flocking rules to simulate realistic group dynamics. Visualization with Matplotlib provided insights into the Boids' movements, while Pygame enabled interactive and real-time simulations.

Feel free to experiment further by tackling the exercises and exploring more advanced features to enhance the simulation.


## <a name="references"></a>9. References

1. Reynolds, C. W. (1987). "Flocks, Herds, and Schools: A Distributed Behavioral Model." *Computer Graphics*, 21(4), 25–34.
2. Pygame Documentation: [https://www.pygame.org/docs/](https://www.pygame.org/docs/)
3. Matplotlib Animation Guide: [https://matplotlib.org/stable/api/animation_api.html](https://matplotlib.org/stable/api/animation_api.html)
4. NumPy Documentation: [https://numpy.org/doc/](https://numpy.org/doc/)


---
## <a name="environment_setup"></a> Setting Up Your Local Python Environment

To continue with **Point 5** (Pygame) of the laboratory, follow these steps to set up Python and the necessary libraries on your computer.

### 1. Install Python

1. **Download Python**:
   - Visit the [Python Downloads](https://www.python.org/downloads/) page.
   - Download the latest version of Python suitable for your operating system (Windows, macOS, or Linux).

2. **Run the Installer**:
   - **Windows**:
     - Double-click the downloaded installer.
     - **Important**: Check the box that says **"Add Python to PATH"** before clicking "Install Now".
   - **macOS**:
     - Open the downloaded `.pkg` file and follow the installation prompts.
   - **Linux**:
     - Python is usually pre-installed. To check, open Terminal and type:
       ```bash
       python3 --version
       ```
     - If not installed, use your package manager. For example, on Ubuntu:
       ```bash
       sudo apt-get update
       sudo apt-get install python3 python3-pip
       ```

### 2. Install an Integrated Development Environment (IDE)

An IDE helps you write and run Python code efficiently. We recommend **Visual Studio Code (VS Code)**.

1. **Download VS Code**:
   - Go to the [Visual Studio Code](https://code.visualstudio.com/) website.
   - Download the installer for your operating system and run it.

2. **Install Python Extension**:
   - Open VS Code.
   - Click on the **Extensions** icon on the left sidebar (or press `Ctrl+Shift+X`).
   - Search for **"Python"** and install the official Python extension by Microsoft.

### 3. Install Required Python Libraries

1. **Open Command Prompt or Terminal**:
   - **Windows**: Press `Win + R`, type `cmd`, and press Enter.
   - **macOS/Linux**: Open the Terminal application.

2. **Install Libraries**:
   - Run the following command to install `numpy`, `matplotlib`, and `pygame`:
     ```bash
     pip install numpy matplotlib pygame
     ```
   - **Note**: If you encounter permission issues, you might need to add `--user`:
     ```bash
     pip install --user numpy matplotlib pygame
     ```

### 4. Verify the Installation

1. **Create a Python Script**:
   - Open VS Code.
   - Create a new file and name it `boids_pygame.py`.

2. **Add Pygame Code**:
   - Copy the Pygame code from **Point 5** of the laboratory and paste it into `boids_pygame.py`.

3. **Run the Script**:
   - Open the integrated terminal in VS Code (`View` > `Terminal`).
   - Navigate to the directory containing `boids_pygame.py` if necessary.
   - Run the script using:
     ```bash
     python boids_pygame.py
     ```
   - A Pygame window should open, displaying the Boids simulation.

### Troubleshooting

- **`pip` Command Not Found**:
  - Ensure Python is added to your system PATH.
  - Reinstall Python and check the **"Add Python to PATH"** option.

- **Pygame Window Not Opening**:
  - Make sure you're running the script on your local machine, not in an online environment like Colab.
  - Check for any error messages in the terminal and ensure all libraries are installed correctly.

- **Permission Errors**:
  - On **Windows**, try running Command Prompt as an administrator.
  - On **macOS/Linux**, you might need to use `sudo`:
    ```bash
    sudo pip install numpy matplotlib pygame
    ```

---
